# AI Art Agent — Colab 免费 GPU 跑 ComfyUI

本 notebook 在 Colab 免费 GPU（T4）上安装 ComfyUI，下载 SD 1.5 模型，并开启公网隧道，
让 AI Art Agent 通过「远程模式」连接生成图片。

使用步骤：
1. 菜单栏选择 代码执行程序 → 更改运行时类型 → T4 GPU
2. 依次运行下方所有代码单元格
3. 把最后一个单元格输出的 `https://xxx.trycloudflare.com` 地址，填进 AI Art Agent 的连接设置（远程模式）

In [ ]:
# 0. 检查 GPU 是否可用
import os
print(os.popen('nvidia-smi -L').read() or '未检测到 NVIDIA GPU')

In [ ]:
# 1. 安装 ComfyUI（Colab 已预装 PyTorch，直接用）
%cd /content
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
import torch
print('CUDA 可用:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 1.5 安装 ComfyUI Manager（提供在线模型库，可选）
%cd /content/ComfyUI
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git custom_nodes/ComfyUI-Manager
!pip install -q -r custom_nodes/ComfyUI-Manager/requirements.txt
print('ComfyUI Manager 已安装')

In [ ]:
# 1.6 安装 AnimateDiff 节点并下载运动模型（视频生成）
%cd /content/ComfyUI
!git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git custom_nodes/ComfyUI-AnimateDiff-Evolved
import os
os.makedirs('/content/ComfyUI/models/animatediff_models', exist_ok=True)
if not os.path.exists('/content/ComfyUI/models/animatediff_models/mm_sd_v15_v2.ckpt'):
    !wget -q https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt -O /content/ComfyUI/models/animatediff_models/mm_sd_v15_v2.ckpt
print('AnimateDiff 已安装，运动模型:', os.path.exists('/content/ComfyUI/models/animatediff_models/mm_sd_v15_v2.ckpt'))

In [ ]:
# 2. 下载 SD 1.5 模型（约 4GB）
# 提示：若已挂载 Google Drive 且存在 MyDrive/ComfyUI/v1-5-pruned-emaonly.safetensors，则复用，不再重复下载
from huggingface_hub import hf_hub_download

checkpoints = '/content/ComfyUI/models/checkpoints'
os.makedirs(checkpoints, exist_ok=True)
drive_model = '/content/drive/MyDrive/ComfyUI/v1-5-pruned-emaonly.safetensors'
if os.path.exists(drive_model):
    os.symlink(drive_model, checkpoints + '/v1-5-pruned-emaonly.safetensors')
    print('复用 Drive 中的模型')
else:
    hf_hub_download(
        repo_id='stable-diffusion-v1-5/stable-diffusion-v1-5',
        filename='v1-5-pruned-emaonly.safetensors',
        local_dir=checkpoints,
    )
    print('模型下载完成')

In [ ]:
# 3. 后台启动 ComfyUI（监听 0.0.0.0:8188）
%cd /content/ComfyUI
!nohup python main.py --listen 0.0.0.0 --port 8188 --disable-auto-launch > /content/comfy.log 2>&1 &
import time
ok = False
for _ in range(20):
    r = !curl -s http://127.0.0.1:8188/system_stats | head -c 200
    if r and 'system' in r[0]:
        ok = True
        break
    time.sleep(3)
print('ComfyUI 已启动' if ok else '启动超时，请查看 /content/comfy.log')

In [ ]:
# 4. 开启公网隧道（cloudflared，无需注册）
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!nohup /usr/local/bin/cloudflared tunnel --url http://127.0.0.1:8188 --no-autoupdate > /content/tunnel.log 2>&1 &
import re
url = None
for _ in range(30):
    log = open('/content/tunnel.log').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log)
    if m:
        url = m.group(0)
        break
    time.sleep(2)
if url:
    print('公网地址：' + url)
    print('在 AI Art Agent 远程模式填入此地址，保存后点“测试连接”')
else:
    print('隧道尚未就绪，请查看 /content/tunnel.log')

In [ ]:
# 可选：改用 ngrok 隧道（需先到 https://dashboard.ngrok.com 注册并复制 Authtoken）
# !pip install -q pyngrok
# from pyngrok import ngrok
# ngrok.set_auth_token('你的_ngrok_authtoken')
# print('公网地址：' + ngrok.connect(8188, bind_tls=True).public_url)

# 使用说明

- 把「步骤 4」输出的地址（形如 `https://xxx.trycloudflare.com`）填到 AI Art Agent 连接设置的远程模式，保存后测试连接。
- 会话关闭或断线后地址失效；下次重跑本 notebook 会得到新地址。
- 免费额度有限，Colab 长时间空闲会被回收，建议测试完及时关闭。
- 若国内网络无法直连 `trycloudflare.com`，改用 ngrok 隧道（见上方可选单元格）。